# Notebook 3 — Fine-Tuning, Threshold, and Ensemble (multi-class → binary)

Hyperband over InceptionResNetV2 (still multi-class softmax over the
36 breeds), then threshold-tune the *binary* decision (sum of restricted
softmax probabilities), then ensemble the top frozen-base models.


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── Verify GPU ────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# ── Project root on Drive ─────────────────────────────────
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/MSc_Capstone')
for d in ['data', 'pretrained/checkpoints', 'pretrained/logs',
          'pretrained/curves', 'pretrained/inference', 'report']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

DATA   = ROOT / 'data'
CKPT   = ROOT / 'pretrained' / 'checkpoints'
LOGS   = ROOT / 'pretrained' / 'logs'
CURVES = ROOT / 'pretrained' / 'curves'
INFER  = ROOT / 'pretrained' / 'inference'
REPORT = ROOT / 'report'

print(f'Project root: {ROOT}')
print(f'Data folder:  {DATA}')


## 3.1 — Install Keras Tuner and setup


In [ ]:
!pip install -q keras-tuner

import keras_tuner as kt
from tensorflow.keras import layers
from tensorflow.keras.applications import inception_resnet_v2
import numpy as np, pandas as pd, json, random
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    matthews_corrcoef, average_precision_score, roc_auc_score
)

SEED = 58
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)

IMG_SIZE   = (256, 256)
BATCH_SIZE = 32
EPOCHS     = 8

print('Ready for hyperparameter search.')


In [ ]:
# ── Load breed manifest written by Notebook 1 ────────────────
import json, numpy as np

with open(DATA / 'breed_to_restricted.json') as f:
    MANIFEST = json.load(f)

SELECTED_BREEDS = MANIFEST['breeds']
RESTRICTED_SET  = set(MANIFEST['restricted'])
NUM_CLASSES     = len(SELECTED_BREEDS)
BREED_TO_IDX    = {b: i for i, b in enumerate(SELECTED_BREEDS)}

# Boolean mask aligned with class index order used by image_dataset_from_directory
# (alphabetical). image_dataset_from_directory sorts class names alphabetically,
# so we mirror that ordering here.
CLASS_NAMES     = sorted(SELECTED_BREEDS)
RESTRICTED_MASK = np.array([(b in RESTRICTED_SET) for b in CLASS_NAMES], dtype=bool)
RESTRICTED_IDX  = np.where(RESTRICTED_MASK)[0]

print(f'{NUM_CLASSES} classes loaded.')
print(f'Restricted indices (in alphabetical order): {RESTRICTED_IDX.tolist()}')


## 3.2 — Dataset loader


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def create_datasets(input_size=(256, 256)):
    aug = tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ])
    train_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'train'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=True, seed=SEED)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'val'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False)
    test_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'test'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False)

    y_train = np.concatenate([y.numpy() for _, y in train_ds])
    cw = compute_class_weight('balanced',
                              classes=np.arange(NUM_CLASSES), y=y_train)
    class_weight = {i: float(w) for i, w in enumerate(cw)}

    train_ds = train_ds.map(lambda x, y: (aug(x, training=True), y))
    A = tf.data.AUTOTUNE
    return (
        train_ds.cache().shuffle(1000).prefetch(A),
        val_ds.cache().prefetch(A),
        test_ds.cache().prefetch(A),
        class_weight,
    )

train_ds, val_ds, test_ds, class_weight = create_datasets()
print('Datasets loaded.')


## 3.3 — HyperModel


In [ ]:
def build_model(hp):
    dropout  = hp.Choice('dropout', [0.2, 0.3, 0.4, 0.5])
    lr       = hp.Choice('lr', [3e-5, 1e-4, 3e-4])
    unfreeze = hp.Choice('unfreeze_pct', [0.0, 0.1, 0.2, 0.3])

    base = inception_resnet_v2.InceptionResNetV2(
        include_top=False, weights='imagenet',
        input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    if unfreeze > 0:
        cut = int(len(base.layers) * (1 - unfreeze))
        for layer in base.layers[cut:]:
            layer.trainable = True

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = inception_resnet_v2.preprocess_input(inputs)
    x = base(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='top1'),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5'),
        ],
    )
    return model

print('HyperModel defined.')


## 3.4 — Hyperband search


In [ ]:
tuner = kt.Hyperband(
    build_model,
    objective='val_top1',
    max_epochs=EPOCHS,
    factor=4,
    directory=str(ROOT / 'pretrained' / 'hyperband'),
    project_name='restricted_dogs_breeds'
)

stop_early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=2, restore_best_weights=True)

print('Starting Hyperband search...\n')
tuner.search(train_ds, validation_data=val_ds, epochs=EPOCHS,
             callbacks=[stop_early], class_weight=class_weight, verbose=2)

best_hp = tuner.get_best_hyperparameters(1)[0]
print('\nBest hyperparameters:')
print(f'  Dropout:       {best_hp.get("dropout")}')
print(f'  Learning rate: {best_hp.get("lr")}')
print(f'  Unfreeze %:    {best_hp.get("unfreeze_pct")}')

best_model = tuner.get_best_models(1)[0]
best_model.save(str(CKPT / 'FineTuned_best.keras'))
print(f'Saved: {CKPT / "FineTuned_best.keras"}')


## 3.5 — Threshold optimisation on the *binary* prediction


In [ ]:
y_true_mc = np.concatenate([y.numpy() for _, y in test_ds])
y_prob_mc = best_model.predict(test_ds).reshape(-1, NUM_CLASSES)
y_prob    = y_prob_mc[:, RESTRICTED_IDX].sum(axis=1)
y_true    = RESTRICTED_MASK[y_true_mc].astype(int)

thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TP, TN, FP, FN = cm[1,1], cm[0,0], cm[0,1], cm[1,0]
    prec = TP / max(TP+FP, 1)
    rec  = TP / max(TP+FN, 1)
    f1   = 2*TP / max(2*TP+FP+FN, 1)
    mcc  = matthews_corrcoef(y_true, y_pred)
    rows.append({'threshold': round(t,2), 'precision': prec,
                 'recall': rec, 'f1': f1, 'mcc': mcc})

df_t = pd.DataFrame(rows)
best_row = df_t.loc[df_t['f1'].idxmax()]
print(f'Optimal threshold: {best_row["threshold"]}  '
      f'F1={best_row["f1"]:.3f}  P={best_row["precision"]:.3f}  '
      f'R={best_row["recall"]:.3f}  MCC={best_row["mcc"]:.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_t['threshold'], df_t['precision'], 'g-', lw=2, label='Precision')
ax.plot(df_t['threshold'], df_t['recall'],    'b-', lw=2, label='Recall')
ax.plot(df_t['threshold'], df_t['f1'],        'r-', lw=2.5, label='F1')
ax.axvline(best_row['threshold'], color='gray', ls='--', alpha=0.7,
           label=f'Optimal ({best_row["threshold"]})')
ax.set_xlabel('Threshold on P(restricted)'); ax.set_ylabel('Score')
ax.set_title('Threshold Optimisation (binary)', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(INFER / 'threshold_optimisation.png'), dpi=300)
plt.show()
df_t.to_csv(str(INFER / 'threshold_sweep.csv'), index=False)


## 3.6 — ROC, PR, and classification report (binary)


In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc_val = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_val:.3f}')
ax.plot([0,1],[0,1], '--', color='gray')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Fine-Tuned (binary view)', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig(str(INFER / 'roc_curve.png'), dpi=300); plt.show()

prec_c, rec_c, _ = precision_recall_curve(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec_c, prec_c, lw=2, label=f'PR-AUC = {pr_auc:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig(str(INFER / 'pr_curve.png'), dpi=300); plt.show()

opt_t = best_row['threshold']
y_opt = (y_prob >= opt_t).astype(int)
report = classification_report(y_true, y_opt,
                               target_names=['Unrestricted', 'Restricted'],
                               output_dict=True, digits=4)
df_report = pd.DataFrame(report).T
print(df_report.round(4))
df_report.to_csv(str(LOGS / 'classification_report.csv'))


## 3.7 — Ensemble: average softmax across the best models, collapse to binary


In [ ]:
frozen_models = sorted(CKPT.glob('*_best.keras'))
frozen_models = [m for m in frozen_models if 'FineTuned' not in m.name]
print(f'Found {len(frozen_models)} frozen-base checkpoints:')
for m in frozen_models:
    print(f'  {m.name}')

# Test set at 299x299 — matches the largest input size in the zoo.
test_ds_ens = tf.keras.utils.image_dataset_from_directory(
    str(DATA / 'test'), image_size=(299, 299), batch_size=32,
    label_mode='int', class_names=CLASS_NAMES, shuffle=False)
y_true_mc_ens  = np.concatenate([y.numpy() for _, y in test_ds_ens])
y_true_bin_ens = RESTRICTED_MASK[y_true_mc_ens].astype(int)

model_softmax = {}
model_bin_acc = {}

for model_path in frozen_models:
    name = model_path.stem.replace('_best', '')
    try:
        m = tf.keras.models.load_model(str(model_path))
        preds = m.predict(test_ds_ens, verbose=0)
        bin_prob = preds[:, RESTRICTED_IDX].sum(axis=1)
        acc = float(np.mean((bin_prob > 0.5).astype(int) == y_true_bin_ens))
        model_softmax[name] = preds
        model_bin_acc[name] = acc
        print(f'  {name}: bin_acc={acc:.4f}')
        del m; tf.keras.backend.clear_session()
    except Exception as e:
        print(f'  {name}: failed — {e}')

ft_path = CKPT / 'FineTuned_best.keras'
if ft_path.exists():
    ft_model = tf.keras.models.load_model(str(ft_path))
    ft_preds = ft_model.predict(test_ds_ens, verbose=0)
    ft_bin = ft_preds[:, RESTRICTED_IDX].sum(axis=1)
    ft_acc = float(np.mean((ft_bin > 0.5).astype(int) == y_true_bin_ens))
    model_softmax['FineTuned'] = ft_preds
    model_bin_acc['FineTuned'] = ft_acc
    print(f'  FineTuned: bin_acc={ft_acc:.4f}')
    del ft_model; tf.keras.backend.clear_session()

ranked = sorted(model_bin_acc.items(), key=lambda x: -x[1])
print('\nRanking (binary accuracy):')
for i, (n, s) in enumerate(ranked, 1):
    print(f'  #{i} {n}: {s:.4f}')

top3 = [n for n, _ in ranked[:3]]
all_n = [n for n, _ in ranked]

ensembles = {}
ensembles['Top3_Average']  = np.mean([model_softmax[n] for n in top3], axis=0)
ensembles['All_Average']   = np.mean([model_softmax[n] for n in all_n], axis=0)
w = np.array([model_bin_acc[n] for n in top3]); w = w / w.sum()
ensembles['Top3_Weighted'] = np.average([model_softmax[n] for n in top3], axis=0, weights=w)

print(f'\n{"="*60}\nENSEMBLE RESULTS (binary view)\n{"="*60}')
ensemble_results = []
for ens_name, ens_softmax in ensembles.items():
    ens_bin = ens_softmax[:, RESTRICTED_IDX].sum(axis=1)
    y_ens   = (ens_bin > 0.5).astype(int)
    cm = confusion_matrix(y_true_bin_ens, y_ens, labels=[0, 1])
    TP, TN, FP, FN = cm[1,1], cm[0,0], cm[0,1], cm[1,0]
    acc  = (TP+TN) / max(TP+TN+FP+FN, 1)
    prec = TP / max(TP+FP, 1)
    rec  = TP / max(TP+FN, 1)
    f1   = 2*TP / max(2*TP+FP+FN, 1)
    mcc  = matthews_corrcoef(y_true_bin_ens, y_ens)
    ens_auc = roc_auc_score(y_true_bin_ens, ens_bin)
    ensemble_results.append({'ensemble': ens_name,
                             'models': ', '.join(top3) if '3' in ens_name else 'all',
                             'accuracy': acc, 'precision': prec, 'recall': rec,
                             'f1': f1, 'mcc': mcc, 'auc': ens_auc})
    print(f'{ens_name}: Acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  '
          f'F1={f1:.4f}  MCC={mcc:.4f}  AUC={ens_auc:.4f}')

best_single = ranked[0]
best_ens = max(ensemble_results, key=lambda x: x['f1'])
print(f'\nBest single model: {best_single[0]} (bin_acc={best_single[1]:.4f})')
print(f'Best ensemble:     {best_ens["ensemble"]} (F1={best_ens["f1"]:.4f})')

pd.DataFrame(ensemble_results).to_csv(str(INFER / 'ensemble_results.csv'), index=False)
print('\nNotebook 3 complete. Open Notebook 4 and Run all.')
